# Long sandwich cylinders under hydrostatic pressure: Kardomateas and Simitses (2003)

G. A. Kardomateas and G. J. Simitses, *Buckling of long sandwich cylindrical shells under external pressure*, in: E. E. Gdoutos and Z. P. Marioli-Riga (eds), Recent Advances in Composite Materials, Kluwer/Springer, Dordrecht, 2003, pp. 291-300, https://doi.org/10.1007/978-94-017-2852-2_24.

The same sandwich rings as Han, Kardomateas and Simitses (2004), see `han2004_sandwich_rings.ipynb`, under a uniform hydrostatic pressure on the outer surface, mode $n = 2$:

- Table 1: the materials, with $G_{13}$ the transverse shear modulus of the faces and of the alloy-foam core (1.72E+04 kPa, the isotropic value $E/(2(1 + \nu)) = 0.017256$ GPa rounded);
- Table 2: faces $h = 2.54$ mm, core $c = 25.4$ mm (the text uses $h$ for the faces), $R/h_{tot}$ = 30, 60 and 120 (the text mentions also 240, not in the table, and the last row of the graphite faces reads 20 for 120), the same values as Table 2 of Han et al. for these ratios;
- Table 3: faces of 0.3 in and core of 0.6 in, graphite/epoxy, the same as Table 3 of Han et al.;
- Table 4: a homogeneous graphite/epoxy ring of the same total thickness $h_{tot} = 30.48$ mm, not in Han et al.

The shell formulas are those of Han et al., Eqs. (1)-(10) here.

In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import follower_utils as fu
from follower_utils import val, fmt, pct, markdown_table

# True runs panels again, otherwise the results saved in RESULTS are loaded
RERUN = False
RESULTS = 'results'

## panels model

A long cylinder under a uniform pressure buckles as a ring, in generalized plane strain, with the mode $w = \cos(n\theta)$, $n = 2$. The ring is modelled by `val.ring_shell`, a quarter of the circumference, $b = \pi R/2$, of a cylindrical shell with Sanders' kinematics:

- symmetry planes at the straight edges, $v = 0$ and $w_{,y} = 0$ (and $\phi_y = 0$ for the FSDT and TSDT), which contain the ring modes $n = 2, 4, \ldots$;
- symmetry planes at the ends, $u = 0$, $v_{,x} = w_{,x} = 0$, i.e. generalized plane strain;
- 4 terms along $x$ and 15 along $y$.

The pressure is a follower pressure, `add_pressure_load(-1., follower=True, cte=False, zp=zp)`, with `zp` the position of the loaded surface: `zp = h/2` places it on the outer surface, as in the elasticity solution, applied as $p(1 + z_p/R)$ per unit mid-surface area (shallow shell). The prestress comes from a linear static analysis, and the critical pressure from `val.ring_buckling_pressure`:

```python
c0 = solve(k0, s.calc_fext())
lb(k0, s.calc_kG(c=c0) + s.calc_kCfollower())
```

The fibres of the faces are along the hoop direction, i.e. plies at 90 deg, with `laminaprop = (E11, E22, nu12, G12, G13, G23) = (E2, E3, nu23, G23, G12, G31)` in the notation of the reference (1 = r, 2 = $\theta$, 3 = z): $G_{13}$ of the ply, the transverse shear stiffness `A44` of the ring, is the $r\theta$ modulus $G_{12}$ of the reference.

In [2]:
RATIOS = (30, 60, 120)


def compute():
    res = {}
    for face in val.FACES:
        for ratio in RATIOS:
            res['%s %d' % (face, ratio)] = fu.sandwich_ring_case(face, ratio)
    for ratio, *_ in val.HAN2004_TABLE3[1:]:
        res['table3 %d' % ratio] = fu.sandwich_ring_case('graphite', ratio, f=0.3*val.INCH, c=0.6*val.INCH)
    # Table 4, homogeneous graphite/epoxy
    mat = val.FACES['graphite']
    for ratio in RATIOS:
        R = ratio*val.H
        out = {}
        for model in ('cylshell_clpt_sanders', 'cylshell_fsdt_sanders', 'cylshell_tsdt_sanders'):
            s = val.ring_shell(model, R, [90.], [val.H], [mat])
            out[model] = val.ring_buckling_pressure(s)
            out[model + ', zp = h/2'] = val.ring_buckling_pressure(s, zp=val.H/2)
        res['homogeneous %d' % ratio] = out
    return res


res = fu.run_or_load(os.path.join(RESULTS, 'kardomateas2003_sandwich_rings.json'), compute, rerun=RERUN)

## Table 2: sandwich with alloy-foam core

Critical pressure in Pa, differences to the 3D elasticity solution in parentheses. The shell columns of the paper are reproduced by panels with the CLPT (classical) and the FSDT with the shear stiffness of each formula (core only, $\bar{G}$), with the pressure on the mid-surface as in the formulas; the last columns are the FSDT with the default shear correction of panels (Rohwer) and the TSDT with the pressure on the outer surface.

In [3]:
rows = []
for face in val.FACES:
    for i, ratio in enumerate(val.RATIOS):
        if ratio not in RATIOS:
            continue
        r = res['%s %d' % (face, ratio)]
        elast, core_only, gbar = val.HAN2004[face][i][:3]
        cl = val.HAN2004_CLASSICAL[face][i]
        rows.append([face, ratio, fmt(elast),
                     '%s / %s' % (fmt(cl, percent=pct(cl, elast)), fmt(r['CLPT'], percent=pct(r['CLPT'], elast))),
                     '%s / %s' % (fmt(core_only, percent=pct(core_only, elast)), fmt(r['FSDT core'], percent=pct(r['FSDT core'], elast))),
                     '%s / %s' % (fmt(gbar, percent=pct(gbar, elast)), fmt(r['FSDT Gbar'], percent=pct(r['FSDT Gbar'], elast))),
                     fmt(r['FSDT rohwer, zp = h/2'], percent=pct(r['FSDT rohwer, zp = h/2'], elast)),
                     fmt(r['TSDT, zp = h/2'], percent=pct(r['TSDT, zp = h/2'], elast))])
display(Markdown(markdown_table(['faces', '$R/h_{tot}$', '3D elasticity', 'classical: paper / panels CLPT',
                                 'shear, $G_c$: paper / panels FSDT', 'shear, $\\bar{G}$: paper / panels FSDT',
                                 'panels FSDT rohwer, zp = h/2', 'panels TSDT, zp = h/2'], rows)))

| faces | $R/h_{tot}$ | 3D elasticity | classical: paper / panels CLPT | shear, $G_c$: paper / panels FSDT | shear, $\bar{G}$: paper / panels FSDT | panels FSDT rohwer, zp = h/2 | panels TSDT, zp = h/2 |
|---|---|---|---|---|---|---|---|
| boron | 30 | 277,305 | 862,343 (+211.0%) / 866,463 (+212.5%) | 253,721 (-8.5%) / 254,077 (-8.4%) | 323,361 (+16.6%) / 323,939 (+16.8%) | 285,846 (+3.1%) | 592,598 (+113.7%) |
| boron | 60 | 70,416 | 107,793 (+53.1%) / 108,327 (+53.8%) | 67,383 (-4.3%) / 67,592 (-4.0%) | 76,087 (+8.1%) / 76,353 (+8.4%) | 71,838 (+2.0%) | 96,818 (+37.5%) |
| boron | 120 | 11,817 | 13,474 (+14.0%) / 13,541 (+14.6%) | 11,717 (-0.8%) / 11,768 (-0.4%) | 12,203 (+3.3%) / 12,258 (+3.7%) | 11,999 (+1.5%) | 13,125 (+11.1%) |
| graphite | 30 | 258,549 | 706,307 (+173.2%) / 709,336 (+174.4%) | 238,236 (-7.9%) / 238,580 (-7.7%) | 298,643 (+15.5%) / 299,183 (+15.7%) | 266,082 (+2.9%) | 535,525 (+107.1%) |
| graphite | 60 | 61,528 | 88,288 (+43.5%) / 88,683 (+44.1%) | 59,207 (-3.8%) / 59,384 (-3.5%) | 65,825 (+7.0%) / 66,044 (+7.3%) | 62,570 (+1.7%) | 81,756 (+32.9%) |
| graphite | 120 | 9,918 | 11,036 (+11.3%) / 11,086 (+11.8%) | 9,829 (-0.9%) / 9,869 (-0.5%) | 10,168 (+2.5%) / 10,211 (+3.0%) | 10,023 (+1.1%) | 10,835 (+9.2%) |
| kevlar | 30 | 171,351 | 296,324 (+72.9%) / 298,794 (+74.4%) | 162,433 (-5.2%) / 163,172 (-4.8%) | 188,347 (+9.9%) / 189,342 (+10.5%) | 174,563 (+1.9%) | 228,673 (+33.5%) |
| kevlar | 60 | 31,418 | 37,040 (+17.9%) / 37,356 (+18.9%) | 30,712 (-2.2%) / 30,928 (-1.6%) | 32,397 (+3.1%) / 32,638 (+3.9%) | 31,639 (+0.7%) | 34,580 (+10.1%) |
| kevlar | 120 | 4,476 | 4,630 (+3.4%) / 4,670 (+4.3%) | 4,403 (-1.6%) / 4,439 (-0.8%) | 4,470 (-0.1%) / 4,507 (+0.7%) | 4,460 (-0.4%) | 4,569 (+2.1%) |

## Table 3: thicker faces, $f = 0.3$ in, $c = 0.6$ in

In [4]:
rows = []
for ratio, elast, classical, core_only, gbar in val.HAN2004_TABLE3[1:]:
    r = res['table3 %d' % ratio]
    rows.append([ratio, fmt(elast),
                 '%s / %s' % (fmt(classical, percent=pct(classical, elast)), fmt(r['CLPT'], percent=pct(r['CLPT'], elast))),
                 '%s / %s' % (fmt(core_only, percent=pct(core_only, elast)), fmt(r['FSDT core'], percent=pct(r['FSDT core'], elast))),
                 '%s / %s' % (fmt(gbar, percent=pct(gbar, elast)), fmt(r['FSDT Gbar'], percent=pct(r['FSDT Gbar'], elast))),
                 fmt(r['FSDT rohwer, zp = h/2'], percent=pct(r['FSDT rohwer, zp = h/2'], elast)),
                 fmt(r['TSDT, zp = h/2'], percent=pct(r['TSDT, zp = h/2'], elast))])
display(Markdown(markdown_table(['$R/h_{tot}$', '3D elasticity', 'classical: paper / panels CLPT',
                                 'shear, $G_c$: paper / panels FSDT', 'shear, $\\bar{G}$: paper / panels FSDT',
                                 'panels FSDT rohwer, zp = h/2', 'panels TSDT, zp = h/2'], rows)))

| $R/h_{tot}$ | 3D elasticity | classical: paper / panels CLPT | shear, $G_c$: paper / panels FSDT | shear, $\bar{G}$: paper / panels FSDT | panels FSDT rohwer, zp = h/2 | panels TSDT, zp = h/2 |
|---|---|---|---|---|---|---|
| 30 | 393,573 | 1,466,490 (+272.6%) / 1,472,828 (+274.2%) | 188,038 (-52.2%) / 188,142 (-52.2%) | 542,378 (+37.8%) / 543,243 (+38.0%) | 378,773 (-3.8%) | 1,386,724 (+252.3%) |
| 60 | 105,699 | 183,311 (+73.4%) / 184,126 (+74.2%) | 67,900 (-35.8%) / 68,011 (-35.7%) | 128,553 (+21.6%) / 128,954 (+22.0%) | 107,021 (+1.3%) | 180,586 (+70.8%) |
| 120 | 19,297 | 22,914 (+18.7%) / 23,016 (+19.3%) | 16,081 (-16.7%) / 16,131 (-16.4%) | 20,709 (+7.3%) / 20,792 (+7.7%) | 19,481 (+1.0%) | 22,857 (+18.4%) |

## Table 4: homogeneous graphite/epoxy ring

$h = 30.48$ mm, fibres along the hoop direction. The shell formulas reduce to $p = 3 E_2 h^3/(12 R^3)/(1 + 4 k_s)$; panels gives the CLPT and, with the pressure on the outer surface, the FSDT (shear correction 5/6 for a homogeneous section) and the TSDT.

In [5]:
KS2003_TABLE4 = [(30, 1641360, 1675930, 1657330, 1660400),
                 (60, 208228, 209491, 208905, 209002),
                 (120, 26180, 26186, 26168, 26171)]
rows = []
for ratio, elast, classical, core_only, gbar in KS2003_TABLE4:
    r = res['homogeneous %d' % ratio]
    p_clpt = r['cylshell_clpt_sanders']
    rows.append([ratio, fmt(elast), fmt(classical, percent=pct(classical, elast)),
                 fmt(core_only, percent=pct(core_only, elast)), fmt(gbar, percent=pct(gbar, elast)),
                 fmt(p_clpt, percent=pct(p_clpt, elast)),
                 fmt(r['cylshell_fsdt_sanders'], percent=pct(r['cylshell_fsdt_sanders'], elast)),
                 fmt(r['cylshell_fsdt_sanders, zp = h/2'], percent=pct(r['cylshell_fsdt_sanders, zp = h/2'], elast)),
                 fmt(r['cylshell_tsdt_sanders, zp = h/2'], percent=pct(r['cylshell_tsdt_sanders, zp = h/2'], elast))])
display(Markdown(markdown_table(['$R/h$', '3D elasticity', 'classical', 'shear, $G_c$', 'shear, $\\bar{G}$',
                                 'panels CLPT', 'panels FSDT', 'panels FSDT, zp = h/2', 'panels TSDT, zp = h/2'], rows)))

| $R/h$ | 3D elasticity | classical | shear, $G_c$ | shear, $\bar{G}$ | panels CLPT | panels FSDT | panels FSDT, zp = h/2 | panels TSDT, zp = h/2 |
|---|---|---|---|---|---|---|---|---|
| 30 | 1,641,360 | 1,675,930 (+2.1%) | 1,657,330 (+1.0%) | 1,660,400 (+1.2%) | 1,683,281 (+2.6%) | 1,664,523 (+1.4%) | 1,637,236 (-0.3%) | 1,637,239 (-0.3%) |
| 60 | 208,228 | 209,491 (+0.6%) | 208,905 (+0.3%) | 209,002 (+0.4%) | 210,425 (+1.1%) | 209,834 (+0.8%) | 208,099 (-0.1%) | 208,099 (-0.1%) |
| 120 | 26,180 | 26,186 (+0.0%) | 26,168 (-0.0%) | 26,171 (-0.0%) | 26,304 (+0.5%) | 26,285 (+0.4%) | 26,176 (-0.0%) | 26,176 (-0.0%) |

## Discussion

- The values of Tables 2 and 3 are those of Han et al. (2004) for $R/h_{tot}$ = 30, 60 and 120, and the comparison of `han2004_sandwich_rings.ipynb` applies: the FSDT of panels with the shear correction of Rohwer and the pressure on the outer surface is within a few percent of the elasticity solution for the thin faces of Table 2, and the TSDT is much too stiff for the soft core.
- **Percentages of the paper.** In the classical column, the differences larger than +100% are printed as the ratio to the elasticity solution: e.g. for the boron faces and $R/h_{tot} = 30$, 862,343 against 277,305 Pa is +211%, printed +310.9% (3.109 times); the tables here give the differences.
- For the homogeneous ring of Table 4 every theory agrees with the elasticity solution: the FSDT and the TSDT of panels with the pressure on the outer surface are within 0.25% of it for all radii, and the CLPT is 0.45% above the classical formula of the paper, the plane-strain factor $1/(1 - \nu_{12}\nu_{21})$ of the bending stiffness that the formula omits. The sandwich construction is what makes the transverse shear, and therefore the shear correction factor, decisive.